# Лабораторная работа №1: землетрясения USGS
Марюшкина Софья и Чуева Арина, поток 1.3. Данные: [USGS Earthquake Catalog API](https://earthquake.usgs.gov/fdsnws/event/1/1). Задание: собрать данные с веб-ресурса, сохранить в SQLite с догрузкой и выполнить не менее пяти разных агрегаций с pandas и SQLAlchemy. Ниже — шесть пар запросов.


## 1. Подготовка
В Colab первая ячейка клонирует ветку лабораторной. При запуске из корня репозитория используется текущая копия.


In [ ]:
from pathlib import Path
import os
import subprocess
import sys

if not Path('usgs_lab1.py').exists():
    subprocess.run(['git', 'clone', '--branch', 'lab1/usgs-earthquakes', 'https://github.com/Sofa-maruysh/pda-lab1-usgs-earthquakes.git', 'lab1-usgs'], check=True)
    os.chdir('lab1-usgs')
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-r', 'requirements.txt'], check=True)
print('Каталог проекта:', Path.cwd())


## 2. Extract → Transform → Load
`sync` запрашивает GeoJSON по страницам, преобразует время и координаты, записывает события в SQLite. Первичный ключ `event_id` и upsert исключают дубли; запись обновляется лишь при более новом времени `updated`. Интервалы `[start, end)` заданы в UTC.


In [ ]:
from datetime import date
from pathlib import Path
from usgs_lab1 import sync

db = Path('data/earthquakes.sqlite')
first = sync(db, date(2025, 1, 1), date(2025, 1, 8), min_magnitude=2.5)
repeat = sync(db, date(2025, 1, 1), date(2025, 1, 8), min_magnitude=2.5)
next_week = sync(db, date(2025, 1, 8), date(2025, 1, 15), min_magnitude=2.5)
print('Первая загрузка:', first)
print('Повтор:', repeat)
print('Догрузка:', next_week)


## 3. Шесть агрегаций двумя способами
1. Количество, средняя и максимальная магнитуда по дням. 2. Распределение по магнитуде. 3. Распределение по глубине. 4. Статистика по сетям. 5. Распределение по часам UTC. 6. Сравнение событий с флагом tsunami и без него. SQLAlchemy выполняет SQL в SQLite; pandas группирует загруженные строки в памяти.


In [ ]:
from IPython.display import display
from sqlalchemy import create_engine
from usgs_lab1 import sql_aggregations, pandas_aggregations

engine = create_engine(f'sqlite:///{db.resolve().as_posix()}')
with engine.connect() as connection:
    sql = sql_aggregations(connection)
    pandas = pandas_aggregations(connection)
engine.dispose()

for name in sql:
    print(name, '— SQLAlchemy')
    display(sql[name])
    print(name, '— pandas')
    display(pandas[name])


## 4. Проверка и сохранение результатов
CSV с суффиксами `_sql` и `_pandas` создаются в `results/`. База и CSV не хранятся в Git; результаты можно пересоздать. Подробные числа и выводы приведены в `REPORT.md`.


In [ ]:
from usgs_lab1 import analyze

print(analyze(db, Path('results')))
assert repeat['inserted_or_updated'] == 0, 'Повторная загрузка создала дубли или изменила записи'
assert len(sql) == len(pandas) == 6
print('Проверка завершена')
